# 🏆 Ensemble & Stacking - Final Submission Pipeline

- **Thành viên:** C (Phạm Hồng Nam) & Toàn Nhóm
- **Mục tiêu:** Tối đa hóa ROC-AUC thông qua đa dạng hóa kiến trúc mô hình (Model Diversity):
  1. **Tầng GBDT tinh chỉnh nội bộ:** LightGBM v2 + XGBoost GPU + CatBoost (CV: 0.94732, Public LB: 0.94760)
  2. **Tầng Đa Mô Hình (Multi-Paradigm Simplex Apex):** Deep Neural Network (RealMLP/TabM/DCNN), GPT-2 Autoregressive Consensus, In-Context Prior (Tri-TabPFN), GP-DVAE và Residual Boosted GBDT v6 (CV: **0.95256**, Public LB: **0.95423**).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == 'C' else Path.cwd()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'C' / 'outputs'
SUB_DIR = ROOT / 'submissions'

train_df = pd.read_csv(DATA_DIR / 'train.csv', usecols=['id', 'is_fraud'])
sub_sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
y_true = train_df['is_fraud'].values

# 1. Các mô hình GBDT nội bộ nhóm
local_models = {
    'LightGBM v2': np.load(ROOT / 'A/outputs/lgbm_a_v2_oof.npy'),
    'XGBoost v1': np.load(ROOT / 'B/outputs/xgb_b_v1_oof.npy'),
    'CatBoost v1': np.load(ROOT / 'C/outputs/cat_c_v1_oof.npy')
}

print('=== 1. Điểm OOF ROC-AUC GBDT nội bộ ===')
for name, pred in local_models.items():
    print(f'  * {name:<15}: {roc_auc_score(y_true, pred):.5f}')


In [ ]:
# 2. Ensemble Rank Average nội bộ
oof_rank_local = (rankdata(local_models['LightGBM v2']) + rankdata(local_models['XGBoost v1']) + rankdata(local_models['CatBoost v1'])) / (3.0 * len(y_true))
test_rank_local = (
    rankdata(np.load(ROOT / 'A/outputs/lgbm_a_v2_test.npy')) +
    rankdata(np.load(ROOT / 'B/outputs/xgb_b_v1_test.npy')) +
    rankdata(np.load(ROOT / 'C/outputs/cat_c_v1_test.npy'))
) / (3.0 * len(sub_sample))

auc_local_ens = roc_auc_score(y_true, oof_rank_local)
print(f'Local Ensemble Rank Average OOF AUC: {auc_local_ens:.5f}')


In [ ]:
# 3. Khai thác Multi-Paradigm Pillars (Deep Neural, GPT-2, Tri-TabPFN, Probit Simplex)
sov_path = DATA_DIR / 'external_sovereign' / 's6e11_sovereign_095423_vault.npz'
if sov_path.exists():
    data_sov = np.load(sov_path)
    print('=== 2. Điểm OOF ROC-AUC Multi-Paradigm Pillars ===')
    for p in ['oof_P1_gpt2_5fold', 'oof_P2_tri_tabpfn', 'oof_P4_deep_neural_gam', 'oof_cv_fortress_0952563']:
        print(f'  * {p:<28}: {roc_auc_score(y_true, data_sov[p]):.6f}')
    
    oof_fortress = data_sov['oof_cv_fortress_0952563']
    print(f'>> Fortress 5/5 Outer Folds OOF AUC: {roc_auc_score(y_true, oof_fortress):.6f} <<')


In [ ]:
# 4. Xuất các bản Submission tối ưu lên Kaggle
# Bản 1: 3-Vertex Simplex Apex Centroid (Public LB: 0.95423)
apex_src = DATA_DIR / 'external_sovereign' / 'sub_sovereign_3vertex_simplex_apex.csv'
if apex_src.exists():
    sub_apex = pd.read_csv(apex_src)
    sub_apex.to_csv(SUB_DIR / 'ens_sovereign_apex.csv', index=False)
    print('Đã xuất: submissions/ens_sovereign_apex.csv (Public LB: 0.95423)')

# Bản 2: Mega Ensemble kết hợp Simplex Apex + Fortress + Local GBDT
if sov_path.exists():
    test_apex = data_sov['test_simplex_3vertex_apex_095423plus']
    test_fortress = data_sov['test_cv_fortress_0952563']
    n = len(test_apex)
    final_rank = 0.85 * (rankdata(test_apex)/n) + 0.10 * (rankdata(test_fortress)/n) + 0.05 * test_rank_local
    sub_mega = sub_sample.copy()
    sub_mega['is_fraud'] = final_rank
    sub_mega.to_csv(SUB_DIR / 'ens_mega_v2.csv', index=False)
    print('Đã xuất: submissions/ens_mega_v2.csv (Public LB: 0.95419)')
